# Setup and namespace validation

## Runtime contract

- **Purpose:** create and validate the Bronze/Silver and Gold namespaces used by downstream notebooks.
- **Inputs:** attached `lh_brz_slv_finance` and `lh_gld_finance` lakehouses.
- **Outputs:** `slv`, `quarantine`, and `audit` schemas in Bronze/Silver; `dim` and `fct` schemas in Gold; an in-memory SAP table contract summary.
- **Failure behavior:** namespace or attachment errors fail immediately; no data tables are written.

Attach both lakehouses before the Spark session starts. Downstream notebooks rely on these names and schemas as stable contracts.

In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable
from functools import reduce
from datetime import datetime, timezone
import uuid

BRZ_SLV_LH = "lh_brz_slv_finance"
GLD_LH = "lh_gld_finance"
BRONZE_ROOT = "Files/bronze/sap_finance"
SLV = f"{BRZ_SLV_LH}.slv"
QTN = f"{BRZ_SLV_LH}.quarantine"
AUD = f"{BRZ_SLV_LH}.audit"
DIM = f"{GLD_LH}.dim"
FCT = f"{GLD_LH}.fct"
SOURCE_SYSTEM = "SAP_S4HANA"
BATCH_RUN_ID = str(uuid.uuid4())
LOAD_TIME_UTC = datetime.now(timezone.utc).replace(tzinfo=None)

spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.autoCompact.enabled", "true")
print(BATCH_RUN_ID, LOAD_TIME_UTC)

In [ ]:
# Initialize Gold Lakehouse namespaces before downstream table writes.
for namespace in ["dim", "fct"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GLD_LH}.{namespace}")

In [ ]:
# This notebook owns the Bronze/Silver Lakehouse namespaces.
for namespace in ["slv", "quarantine", "audit"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")

TABLE_CONFIG = {
    "acdoca": {
        "required": ["RBUKRS", "GJAHR", "BELNR", "DOCLN", "RACCT"],
        "keys": ["RCLNT", "RLDNR", "RBUKRS", "GJAHR", "BELNR", "DOCLN"],
        "types": {
            "GJAHR": "int",
            "POPER": "int",
            "BUDAT": "date",
            "BLDAT": "date",
            "HSL": "decimal(23,2)",
            "KSL": "decimal(23,2)",
            "TSL": "decimal(23,2)",
            "WSL": "decimal(23,2)",
            "MSL": "decimal(23,3)",
        },
    },
    "bkpf": {
        "required": ["BUKRS", "GJAHR", "BELNR"],
        "keys": ["MANDT", "BUKRS", "GJAHR", "BELNR"],
        "types": {
            "GJAHR": "int",
            "MONAT": "int",
            "BUDAT": "date",
            "BLDAT": "date",
            "CPUDT": "date",
        },
    },
    "t001": {
        "required": ["BUKRS"],
        "keys": ["MANDT", "BUKRS"],
        "types": {},
    },
    "csks": {
        "required": ["KOKRS", "KOSTL"],
        "keys": ["MANDT", "KOKRS", "KOSTL"],
        "types": {"DATAB": "date", "DATBI": "date"},
    },
    "cepc": {
        "required": ["KOKRS", "PRCTR"],
        "keys": ["MANDT", "KOKRS", "PRCTR"],
        "types": {"DATAB": "date", "DATBI": "date"},
    },
    "ska1": {
        "required": ["KTOPL", "SAKNR"],
        "keys": ["MANDT", "KTOPL", "SAKNR"],
        "types": {},
    },
    "skat": {
        "required": ["SPRAS", "KTOPL", "SAKNR"],
        "keys": ["MANDT", "SPRAS", "KTOPL", "SAKNR"],
        "types": {},
    },
    "lfa1": {
        "required": ["LIFNR"],
        "keys": ["MANDT", "LIFNR"],
        "types": {},
    },
    "kna1": {
        "required": ["KUNNR"],
        "keys": ["MANDT", "KUNNR"],
        "types": {},
    },
}

business_key_rows = [
    (table_name, ",".join(config["keys"]))
    for table_name, config in TABLE_CONFIG.items()
]
display(
    spark.createDataFrame(
        business_key_rows,
        ["table_name", "business_keys"],
    )
)

In [ ]:
print("Schemas visible in the default Bronze/Silver Lakehouse")
spark.sql("SHOW SCHEMAS").show(truncate=False)
